# ゼロから作る Deep Learning ❸ 輪読会
## 第4ステージ「ニューラルネットワークを作る」 ― ステップ 37 〜 41

### これまでの内容 (第 1 〜 3 ステージ)
DeZero の機能を少しずつ拡張し、次のことができるようになった。  

- 自動微分 (`Variable`/`Function`、`y.backward()`)

- 自然なコード (演算子のオーバーロード、`a*b+c`)

- 高階微分 (`create_graph=True` で「逆伝播の逆伝播」)

ただし、これまで扱ってきたのは主にスカラ (0 次元の数値) だった。  
これは、$y = x^2$ のような 1 つの数の計算に対応する。  

### 第 4 ステージのゴール ― テンソルを扱う
ニューラルネットワークは、大量の数を行列やテンソル (多次元配列) としてまとめて計算する。  
そこで、第 4 ステージでは、DeZero を テンソルを扱えるように整備し、最終的にニューラルネットワークを構築できるようにする。  

このブロック (37〜41) では、そのための基本的な関数を実装する。  

| ステップ | テーマ | 内容 |
|---|---|---|
| 37 | テンソルを扱う | スカラ用の実装がテンソルでもそのまま動くことを確認 |
| 38 | 形状を変える関数 | `reshape` (形を変える) と `transpose` (転置) |
| 39 | 和を求める関数 | `sum` (合計) と、その逆伝播 |
| 40 | ブロードキャストを行う関数 | `broadcast_to` / `sum_to`、四則演算のブロードキャストに対応 |
| 41 | 行列の積・アフィン変換 | `matmul` (行列積) と `linear` (全結合層の計算)  |

この範囲の終わりには、DeZero で全結合層 ($y = xW + b$) が書けるようになる。  
これはニューラルネットワークの最も基本的な要素である。  

---

### テンソルの用語について
この回で頻出する用語を先に整理する。  
- スカラ：1 つの数 (例：`5`)。0 次元。
- ベクトル (1 次元)：数を 1 列に並べたもの (例：`[1,2,3]`)。
- 行列 (2 次元)：数を格子状に並べたもの (例：`[[1,2],[3,4]]`)。
- テンソル (任意の次元の多次元配列)：これらをまとめた呼び方。
- 形状 (shape)：各次元の大きさ。例：`(2, 3)` は「2行3列」。


## 準備：高階微分対応の DeZero を読み込む

このノートブックは単独で動くように、第 3 ステージまでの高階微分対応 DeZero (`grad` が `Variable`) を最初に読み込む。  

この上に、テンソルを扱う関数を追加する。なお、この回では `Variable` に `reshape`/`T`/`sum` などのメソッドも追加するので、`Variable` クラスにはそのためのメソッドをあらかじめ用意している。  


In [ ]:
import numpy as np
import weakref
import contextlib
import matplotlib.pyplot as plt

# ===== 第3ステージまでの高階微分対応 DeZero =====
class Config:
    enable_backprop = True

@contextlib.contextmanager
def using_config(name, value):
    old_value = getattr(Config, name)
    setattr(Config, name, value)
    try:
        yield
    finally:
        setattr(Config, name, old_value)

def no_grad():
    return using_config('enable_backprop', False)

def as_array(x):
    if np.isscalar(x):
        return np.array(x)
    return x

def as_variable(obj):
    if isinstance(obj, Variable):
        return obj
    return Variable(obj)


class Variable:
    __array_priority__ = 200

    def __init__(self, data, name=None):
        if data is not None and not isinstance(data, np.ndarray):
            raise TypeError('{} is not supported'.format(type(data)))
        self.data = data
        self.name = name
        self.grad = None
        self.creator = None
        self.generation = 0

    def set_creator(self, func):
        self.creator = func
        self.generation = func.generation + 1

    def cleargrad(self):
        self.grad = None

    @property
    def shape(self):
        return self.data.shape
    @property
    def ndim(self):
        return self.data.ndim
    @property
    def size(self):
        return self.data.size
    @property
    def dtype(self):
        return self.data.dtype

    def __len__(self):
        return len(self.data)

    def __repr__(self):
        if self.data is None:
            return 'variable(None)'
        p = str(self.data).replace('\n', '\n' + ' ' * 9)
        return 'variable(' + p + ')'

    # --- 以下は今回のステップで中身を実装する関数への準備 ---
    def reshape(self, *shape):
        # x.reshape(2,3) でも x.reshape((2,3)) でも動くようにする
        if len(shape) == 1 and isinstance(shape[0], (tuple, list)):
            shape = shape[0]
        return reshape(self, shape)

    @property
    def T(self):
        return transpose(self)

    def sum(self, axis=None, keepdims=False):
        return sum(self, axis, keepdims)

    def backward(self, retain_grad=False, create_graph=False):
        if self.grad is None:
            self.grad = Variable(np.ones_like(self.data))
        funcs = []
        seen_set = set()
        def add_func(f):
            if f not in seen_set:
                funcs.append(f)
                seen_set.add(f)
                funcs.sort(key=lambda x: x.generation)
        add_func(self.creator)
        while funcs:
            f = funcs.pop()
            gys = [output().grad for output in f.outputs]
            with using_config('enable_backprop', create_graph):
                gxs = f.backward(*gys)
                if not isinstance(gxs, tuple):
                    gxs = (gxs,)
                for x, gx in zip(f.inputs, gxs):
                    if x.grad is None:
                        x.grad = gx
                    else:
                        x.grad = x.grad + gx
                    if x.creator is not None:
                        add_func(x.creator)
            if not retain_grad:
                for y in f.outputs:
                    y().grad = None


class Function:
    def __call__(self, *inputs):
        inputs = [as_variable(x) for x in inputs]
        xs = [x.data for x in inputs]
        ys = self.forward(*xs)
        if not isinstance(ys, tuple):
            ys = (ys,)
        outputs = [Variable(as_array(y)) for y in ys]
        if Config.enable_backprop:
            self.generation = max([x.generation for x in inputs])
            for output in outputs:
                output.set_creator(self)
            self.inputs = inputs
            self.outputs = [weakref.ref(output) for output in outputs]
        return outputs if len(outputs) > 1 else outputs[0]
    def forward(self, xs):
        raise NotImplementedError()
    def backward(self, gys):
        raise NotImplementedError()


# --- 四則演算・累乗 (第 2〜3 ステージで実装済み。ブロードキャスト対応は step40 で追加) ---
class Mul(Function):
    def forward(self, x0, x1):
        return x0 * x1
    def backward(self, gy):
        x0, x1 = self.inputs
        return gy * x1, gy * x0
class Neg(Function):
    def forward(self, x):
        return -x
    def backward(self, gy):
        return -gy
class Pow(Function):
    def __init__(self, c):
        self.c = c
    def forward(self, x):
        return x ** self.c
    def backward(self, gy):
        x, = self.inputs
        c = self.c
        return c * x ** (c - 1) * gy

def mul(x0, x1):
    return Mul()(x0, as_array(x1))
def neg(x):
    return Neg()(x)
def pow(x, c):
    return Pow(c)(x)

Variable.__mul__ = mul
Variable.__rmul__ = mul
Variable.__neg__ = neg
Variable.__pow__ = pow


---
# ステップ 37：テンソルを扱う

## 37.1 これまでの実装はテンソルでも動く

これまでスカラ用に書いてきた DeZero は中身が NumPy の演算であるため、テンソルでもほとんどそのまま動く。  
つまり、NumPy は要素ごとの演算 (element-wise) を自動的に計算してくれる。  

例として、テンソル (行列) に対して要素ごとの計算をしてみる。  
まず `sin` と `add` (ブロードキャスト非対応) を用意して、行列に適用する。  


In [ ]:
# テンソル対応の確認用に sin と add を用意
class Sin(Function):
    def forward(self, x):
        return np.sin(x)
    def backward(self, gy):
        x, = self.inputs
        return gy * cos(x)
class Cos(Function):
    def forward(self, x):
        return np.cos(x)
    def backward(self, gy):
        x, = self.inputs
        return gy * -sin(x)
def sin(x):
    return Sin()(x)
def cos(x):
    return Cos()(x)

# 行列 (2x3 テンソル) に sin を適用
x = Variable(np.array([[1, 2, 3],
                       [4, 5, 6]]))
y = sin(x)     # 各要素に sin が適用される (element-wise)
print("x の形状:", x.shape)
print("y = sin(x):")
print(y)


行列の各要素に `sin` が適用された。順伝播が要素ごとに正しく行われている。  

## 37.2 テンソルを使ったときのバックプロパゲーション

逆伝播もテンソルでそのまま動く。  

これまで、逆伝播で伝わる微分 (`grad`) はスカラだった。テンソルの場合、微分は「入力と同じ形状のテンソル」 になる。  

数学的には、出力 $y$ の各要素を入力 $x$ の各要素で微分したものを考える。要素ごとの計算 ($y_{ij} = \sin(x_{ij})$ のように、各要素が独立に決まる計算) では、$x_{ij}$ は $y_{ij}$ にしか影響しない。だから微分も要素ごとに独立に計算でき、`grad` は入力と同じ形状になる。  

> まとめ：要素ごとの計算では、「$x$ の各要素の微分 = 対応する $y$ の要素からの微分」であり、他の要素は無関係。だから `x.grad` は `x` と同じ形状になり、各要素が独立に求まる。  

実際に、行列に対して逆伝播してみる。`grad` が入力と同じ 2 × 3 の形状になることを確認する。  


In [ ]:
x = Variable(np.array([[1, 2, 3],
                       [4, 5, 6]]))
c = Variable(np.array([[10, 20, 30],
                       [40, 50, 60]]))
# t = x * c (要素ごとの積), y = sum(t) ... の代わりに、ここは要素積だけ見る
t = x * c        # 要素ごとの積
print("t = x * c:")
print(t)
print()

# 逆伝播 (t の各要素の微分を 1 として流す)
t.backward(retain_grad=True)
print("x.grad の形状：", x.grad.shape, " ← 入力 x と同じ形状")
print("x.grad:")
print(x.grad)
print()

`x.grad` は `x` と同じ 2 × 3 の形状で、各要素が `c` の対応する値 (掛け算の逆伝播 ＝ 相手の値) になっている。テンソルでも、スカラのときと同じ理屈で逆伝播が動いている。  

> ステップ 37 のまとめ
>
> - これまでのスカラ用実装は、要素ごとの計算ならテンソルでもそのまま動く。  
>
> - テンソルの微分 (`grad`) は「入力と同じ形状」になる。  
>
> - 要素ごとの計算では各要素が独立なので、微分も要素ごとに求まる。  

---
# ステップ 38：形状を変える関数

ニューラルネットワークでは、テンソルの形状を変える操作が頻繁に使われる。ここでは 2 つの機能を実装する。  

- `reshape`：要素はそのままに、形状だけを変える (例：2 × 3 → 6 や 3 × 2)。  

- `transpose`：行列の転置 (行と列を入れ替える)。  

## 38.1 reshape 関数

`reshape` は「要素の中身と総数は変えず、並べ方 (形) だけ変える」操作にあたる。たとえば $2\times3$ の行列 (6 要素) を、長さ 6 のベクトルにしたり、$3\times2$ の行列にしたりできる。  

逆伝播はどうなるのか？  

`reshape` は要素を並べ替えるだけで、値を一切変えない。したがって、逆伝播では上流から来た微分を元の形に戻せばいい (reshape し直す) だけ。  

$$ \text{順伝播：形状} S_{\text{in}} \to S_{\text{out}}, \qquad \text{逆伝播：形状} S_{\text{out}} \to S_{\text{in}} $$

つまり、`Reshape` の `backward` は「入力の形状に `reshape` して返す」だけ。  


In [ ]:
class Reshape(Function):
    def __init__(self, shape):
        self.shape = shape        # 変換後の形状を覚えておく
    def forward(self, x):
        self.x_shape = x.shape    # 入力の形状を覚える (逆伝播で戻すため)
        y = x.reshape(self.shape) # NumPy の reshape で形を変える
        return y
    def backward(self, gy):
        # 逆伝播：上流の微分を、元の入力の形状に戻すだけ
        return reshape(gy, self.x_shape)

def reshape(x, shape):
    if x.shape == shape:          # 形状が同じなら何もしない
        return as_variable(x)
    return Reshape(shape)(x)

# 動作確認: 2x3 → 6 に変形
x = Variable(np.array([[1, 2, 3],
                       [4, 5, 6]]))
y = reshape(x, (6,))
print("reshape 前:", x.shape, " → 後:", y.shape)
print("y:", y)

# 逆伝播: x.grad は元の 2x3 に戻る
y.backward(retain_grad=True)
print("x.grad の形状:", x.grad.shape, " ← 元の 2x3 に戻る")
print("x.grad:")
print(x.grad)


## 38.2 Variable のメソッドとして使う

`reshape(x, (6,))` と書く代わりに、NumPy のように `x.reshape(6)` と書けると使いやすい。  
冒頭の `Variable` クラスに `reshape` メソッドを用意してあるので、すでに使える。  


In [ ]:
x = Variable(np.random.randn(1, 2, 3))    # 形状 (1,2,3)
y1 = x.reshape((2, 3))    # タプルで指定
y2 = x.reshape(2, 3)      # 引数を並べて指定 (NumPy と同じ書き方)
print("x:", x.shape, " → y1:", y1.shape, ", y2:", y2.shape)


## 38.3 行列の転置 (transpose)

転置は、行列の行と列を入れ替える操作である。$m \times n$ の行列が $n \times m$ になる。数学では $A^\top$ と書く。  

$$ A = \begin{pmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \end{pmatrix} \quad\Rightarrow\quad A^\top = \begin{pmatrix} 1 & 4 \\ 2 & 5 \\ 3 & 6 \end{pmatrix} $$

逆伝播は、`reshape` と同じ発想。転置も値を変えず並べ替えるだけなので、逆伝播ではもう一度転置して元に戻すだけ。  


In [ ]:
class Transpose(Function):
    def forward(self, x):
        y = np.transpose(x)    # NumPy の転置
        return y
    def backward(self, gy):
        # 逆伝播：もう一度転置すれば元の形に戻る
        return transpose(gy)

def transpose(x):
    return Transpose()(x)

# 動作確認
x = Variable(np.array([[1, 2, 3],
                       [4, 5, 6]]))
y = transpose(x)     # x.T でも同じ (Variable に T プロパティを用意済み)
print("転置前:", x.shape)
print("転置後:", y.shape)
print(y)

y.backward()
print("x.grad の形状:", x.grad.shape, " ← 元の 2x3")


`x.T` という書き方もできる (NumPy と同じ)。冒頭の `Variable` に `T` プロパティを用意している。  

In [ ]:
x = Variable(np.array([[1, 2, 3],
                       [4, 5, 6]]))
print("x.T:")
print(x.T)     # transpose(x) と同じ


> ステップ 38 のまとめ
>
> - `reshape`：形状だけ変える。逆伝播は「元の形状に戻す」だけ。
>
> - `transpose`：行と列を入れ替える。逆伝播は「もう一度転置」。
>
> - どちらも値は変えず並べ替えるだけなので、逆伝播も並べ替えを戻すだけ。
>
> - `x.reshape(...)` / `x.T` と NumPy 風に書ける。


---
# ステップ 39：和を求める関数

`sum` (合計) を実装する。ニューラルネットワークでは、損失を合計する場面などで頻繁に使う。実は `sum` の逆伝播には、面白い仕組みがある。  

## 39.1 sum 関数の逆伝播

まず、簡単な例で考える。$y = x_0 + x_1$ (2 つの要素の和) のような足し算の逆伝播は「上流の微分をそのまま各入力へ流す」という操作に対応していた。  

$$ y = x_0 + x_1, \qquad \frac{\partial y}{\partial x_0} = 1,\quad \frac{\partial y}{\partial x_1} = 1 $$

`sum` は、たくさんの要素の足し算である。たとえば $y = x_0 + x_1 + \cdots + x_n$。この各要素についての微分は、すべて 1 になる。  

したがって `sum` の逆伝播は、上流から来た 1 つの微分 `gy` を、入力の全要素に配る (コピーする) ことになる。  

数式のイメージ：  

$$ \text{入力 } (x_0, x_1, \ldots, x_n) \xrightarrow{\text{sum}} y \quad\Rightarrow\quad \text{逆伝播: } gy \to (gy, gy, \ldots, gy) $$

「1 つの値を全要素に適用する」＝ ブロードキャスト (後のステップで取り上げる) にあたる。実装では、上流の微分 `gy` を入力と同じ形状に拡張する (`broadcast_to`) ことで実現する。  

> まとめ：`sum` (合計) の逆伝播は「1 つの微分を全要素に適用する」。これは `broadcast_to` (同じ値を拡張する操作) そのもの。「和」と「全体への適用」が対応関係にある。  

まず `broadcast_to` と、それに対応する `sum_to` を用意する (詳細は次ステップ)。ここでは、`sum` の逆伝播に使うため先に定義する。  


In [ ]:
# --- broadcast_to：x を指定した形状に拡張 (同じ値をコピー)---
class BroadcastTo(Function):
    def __init__(self, shape):
        self.shape = shape
    def forward(self, x):
        self.x_shape = x.shape
        y = np.broadcast_to(x, self.shape)   # NumPy で形状を広げる
        return y
    def backward(self, gy):
        return sum_to(gy, self.x_shape)      # 逆伝播は sum_to (広げた分を足し戻す)
def broadcast_to(x, shape):
    if x.shape == shape:
        return as_variable(x)
    return BroadcastTo(shape)(x)

# --- sum_to：x を指定した形状になるまで和をとる ---
def numpy_sum_to(x, shape):
    # x を shape の形になるように、余分な軸で和をとる
    ndim = len(shape)
    lead = x.ndim - ndim
    lead_axis = tuple(range(lead))
    axis = tuple([i + lead for i, sx in enumerate(shape) if sx == 1])
    y = x.sum(lead_axis + axis, keepdims=True)
    if lead > 0:
        y = y.squeeze(lead_axis)
    return y

class SumTo(Function):
    def __init__(self, shape):
        self.shape = shape
    def forward(self, x):
        self.x_shape = x.shape
        y = numpy_sum_to(x, self.shape)
        return y
    def backward(self, gy):
        return broadcast_to(gy, self.x_shape)  # 逆伝播は broadcast_to
def sum_to(x, shape):
    if x.shape == shape:
        return as_variable(x)
    return SumTo(shape)(x)


## 39.2 sum 関数の実装

`sum` を実装します。逆伝播は、上で用意した `broadcast_to` を使い、上流の微分 `gy` を入力の形状に広げます。


In [ ]:
class Sum(Function):
    def __init__(self, axis, keepdims):
        self.axis = axis           # どの軸で和をとるか (後述)
        self.keepdims = keepdims   # 次元を保つか (後述)
    def forward(self, x):
        self.x_shape = x.shape     # 入力の形状を覚える
        y = x.sum(axis=self.axis, keepdims=self.keepdims)
        return y
    def backward(self, gy):
        # 逆伝播：gy を入力の形状に広げる (＝全要素に配る)
        gy = reshape_sum_backward(gy, self.x_shape, self.axis, self.keepdims)
        gx = broadcast_to(gy, self.x_shape)
        return gx

# axis/keepdims 指定時に gy の形を整える補助関数 (詳細は後述の実験で扱う)
def reshape_sum_backward(gy, x_shape, axis, keepdims):
    ndim = len(x_shape)
    tupled_axis = axis
    if axis is None:
        tupled_axis = None
    elif not isinstance(axis, tuple):
        tupled_axis = (axis,)
    if not (ndim == 0 or tupled_axis is None or keepdims):
        actual_axis = [a if a >= 0 else a + ndim for a in tupled_axis]
        shape = list(gy.shape)
        for a in sorted(actual_axis):
            shape.insert(a, 1)
    else:
        shape = gy.shape
    return gy.reshape(shape)

def sum(x, axis=None, keepdims=False):
    return Sum(axis, keepdims)(x)

# 動作確認：すべての要素の和
x = Variable(np.array([[1, 2, 3],
                       [4, 5, 6]]))
y = sum(x)
print("sum(x) =", y.data, " (1+2+3+4+5+6 = 21)")

y.backward()
print("x.grad の形状：", x.grad.shape, " ← 入力と同じ 2x3")
print("x.grad：")
print(x.grad)


`sum` の逆伝播で、`x.grad` が全要素 `1` の 2×3 行列になった。「和の微分は全要素に適用」を確かめた。  

## 39.3 axis と keepdims

NumPy の `sum` と同じく、`axis` (どの軸で和をとるか) と `keepdims` (次元を保つか) を指定できる。  

- `axis=0`：縦方向 (行方向) に和をとる → 各列の合計。形状 `(2,3)` → `(3,)`。  
- `axis=1`：横方向 (列方向) に和をとる → 各行の合計。形状 `(2,3)` → `(2,)`。  
- `keepdims=True`：和をとった軸を残す (大きさ 1 にする)。形状 `(2,3)` → `(1,3)` など。  

> axis の直感：「`axis=0` で和をとる」＝「0 番目の軸 (行) を潰して消す」。3 行あったものが 1 つに集約され、その軸が消える。  


In [ ]:
x = Variable(np.array([[1, 2, 3],
                       [4, 5, 6]]))

y0 = sum(x, axis=0)     # 各列の合計：[1+4, 2+5, 3+6] = [5, 7, 9]
print("axis=0 (各列の合計):", y0.data, " 形状:", y0.shape)

y1 = sum(x, axis=1)     # 各行の合計：[1+2+3, 4+5+6] = [6, 15]
print("axis=1 (各行の合計):", y1.data, " 形状:", y1.shape)

yk = sum(x, axis=0, keepdims=True)   # 軸を残す
print("axis=0, keepdims=True:", yk.data, " 形状:", yk.shape, " ← (1,3)")


> ステップ 39 のまとめ
> - `sum` は要素の合計を求め、各要素の微分はすべて 1。  
>
> - `sum` の逆伝播は「上流の微分を全要素に分配する」＝ `broadcast_to`。  
>
> - 「和」と「ばらまき (broadcast)」が対応関係。  
>
> - `axis` (和をとる軸) と `keepdims` (次元を保つ) を指定できる。  


---
# ステップ 40：ブロードキャストを行う関数

前のステップで `broadcast_to` と `sum_to` を用意した。このステップでは、それらの関係を整理し、四則演算をブロードキャストに対応させる。これで、形の違うテンソルどうしの計算が正しく微分できるようになる。  

## 40.1 broadcast_to と sum_to は「逆の関係」

2 つの関数の役割を整理する。  

- `broadcast_to(x, shape)`：`x` を指定した形状に拡張する (同じ値をコピーして増やす)。例：`[1,2,3]` (形状`(3,)`) → `[[1,2,3],[1,2,3]]` (形状`(2,3)`)。  
- `sum_to(x, shape)`：`x` を指定した形状になるまで和をとる (余分な軸を合計でまとめる)。例：`[[1,2,3],[4,5,6]]` (形状`(2,3)`) → `[[5,7,9]]` (形状`(1,3)`)。  

この 2 つは互いに逆の関係にある。`broadcast_to` の逆伝播は `sum_to`、`sum_to` の逆伝播は `broadcast_to` に対応する。  

$$ \text{broadcast\_to} \xleftrightarrow{\text{逆伝播}} \text{sum\_to} $$

なぜ逆になるのかを説明すると、`broadcast_to` は「1 つの値を複数箇所にコピー」するもので、コピー (分岐) の逆伝播は、第 2 ステージで学んだとおり「コピー先の微分をすべて足し合わせる」だった。つまり、`broadcast_to` の逆は「和をとる」＝ `sum_to` に対応する。  


In [ ]:
# broadcast_to と sum_to の動作を並べて確認
x = Variable(np.array([1, 2, 3]))
b = broadcast_to(x, (2, 3))     # (3,) → (2,3) に広げる
print("broadcast_to([1,2,3], (2,3)):")
print(b)
print()

x2 = Variable(np.array([[1, 2, 3],
                        [4, 5, 6]]))
s = sum_to(x2, (1, 3))          # (2,3) → (1,3) に和をとる
print("sum_to([[1,2,3],[4,5,6]], (1,3)):")
print(s, " ← 各列の合計 [1+4, 2+5, 3+6]")


## 40.3 ブロードキャストへの対応 (四則演算)

NumPy は、形の違う配列どうしの計算をブロードキャスト (自動で形を合わせる) で処理する。  
たとえば：

```python
np.array([1,2,3]) + np.array([10])   # → [11,12,13]
```

これは、`[10]` が `[10,10,10]` に自動で広げられてから足し算されている。  

DeZero の順伝播は NumPy に任せているので、この計算自体はできる。しかし逆伝播で問題が起こる。つまり、ブロードキャストで形が変わると微分の形が合わなくなる。  

### 問題と解決

たとえば `y = x0 + x1` で `x0` が形状 `(3,)`、`x1` が形状 `(1,)` の場合を考える。順伝播では `x1` が `(3,)` に拡張される。逆伝播で `x1` に伝わる微分は形状 `(3,)` になりるが、本来 `x1` の形状は `(1,)` であるため、形状が合わない。  

解決策は逆伝播時に各入力の元の形状に `sum_to` で戻すことである。広げられた分を足し合わせて元の形に戻す。`Add` の `backward` を、入力形状が違うときだけ `sum_to` するように改良する。  


In [ ]:
# ブロードキャスト対応の四則演算を定義
class Add(Function):
    def forward(self, x0, x1):
        self.x0_shape, self.x1_shape = x0.shape, x1.shape  # 各入力の形状を覚える
        y = x0 + x1
        return y
    def backward(self, gy):
        gx0, gx1 = gy, gy
        # 形状が違う (＝ ブロードキャストが起きた) なら、元の形状に和で戻す
        if self.x0_shape != self.x1_shape:
            gx0 = sum_to(gx0, self.x0_shape)
            gx1 = sum_to(gx1, self.x1_shape)
        return gx0, gx1

class Sub(Function):
    def forward(self, x0, x1):
        self.x0_shape, self.x1_shape = x0.shape, x1.shape
        return x0 - x1
    def backward(self, gy):
        gx0, gx1 = gy, -gy
        if self.x0_shape != self.x1_shape:
            gx0 = sum_to(gx0, self.x0_shape)
            gx1 = sum_to(gx1, self.x1_shape)
        return gx0, gx1

class Div(Function):
    def forward(self, x0, x1):
        self.x0_shape, self.x1_shape = x0.shape, x1.shape
        return x0 / x1
    def backward(self, gy):
        x0, x1 = self.inputs
        gx0 = gy / x1
        gx1 = gy * (-x0 / x1 ** 2)
        if self.x0_shape != self.x1_shape:
            gx0 = sum_to(gx0, self.x0_shape)
            gx1 = sum_to(gx1, self.x1_shape)
        return gx0, gx1

# Mul もブロードキャスト対応に更新
class Mul(Function):
    def forward(self, x0, x1):
        self.x0_shape, self.x1_shape = x0.shape, x1.shape
        return x0 * x1
    def backward(self, gy):
        x0, x1 = self.inputs
        gx0 = gy * x1
        gx1 = gy * x0
        if self.x0_shape != self.x1_shape:
            gx0 = sum_to(gx0, self.x0_shape)
            gx1 = sum_to(gx1, self.x1_shape)
        return gx0, gx1

def add(x0, x1):
    return Add()(x0, as_array(x1))
def sub(x0, x1):
    return Sub()(x0, as_array(x1))
def rsub(x0, x1):
    return Sub()(as_array(x1), x0)
def div(x0, x1):
    return Div()(x0, as_array(x1))
def rdiv(x0, x1):
    return Div()(as_array(x1), x0)
def mul(x0, x1):
    return Mul()(x0, as_array(x1))

Variable.__add__ = add
Variable.__radd__ = add
Variable.__sub__ = sub
Variable.__rsub__ = rsub
Variable.__truediv__ = div
Variable.__rtruediv__ = rdiv
Variable.__mul__ = mul
Variable.__rmul__ = mul


実際に、形の違うテンソルどうしの足し算とその逆伝播を確認する。`x1` (形状 `(1,)`) の微分が、正しく元の形状 `(1,)` になり、値は広げられた分の合計 (3) になることを確かめる。  


In [ ]:
x0 = Variable(np.array([1, 2, 3]))    # 形状 (3,)
x1 = Variable(np.array([10]))         # 形状 (1,)
y = x0 + x1                           # ブロードキャストで [11,12,13]
print("y = x0 + x1 =", y.data)

y.backward()
print("x0.grad：", x0.grad.data, " 形状：", x0.grad.shape, " ← (3,)")
print("x1.grad：", x1.grad.data, " 形状：", x1.grad.shape, " ← (1,) に戻り、値は 1+1+1=3")

> ステップ 40 のまとめ
> - `broadcast_to` (広げる) と `sum_to` (和を取って縮小) は互いに逆の関係。  
>
> - ブロードキャストは「コピー (分岐)」なので、逆伝播は「和」にあたる。  
>
> - 四則演算の `backward` で、入力形状が違うときは `sum_to` で元の形に戻す。  
>
> - これで形の違うテンソルの計算も正しく微分できる。  


---
# ステップ 41：行列の積・アフィン変換

いよいよ、ニューラルネットワークの中心的な機能である行列の積 (MatMul) とアフィン変換 ($y = xW + b$) を実装する。これができれば、全結合層が作れる。  

## 41.1 行列の積の復習

行列の積 $y = xW$ は、ニューラルネットワークで「入力に重みをかける」基本な操作にあたる。形状に注目すると、$x$ が $N \times D$、$W$ が $D \times H$ のとき、積 $y$ は $N \times H$ になる。  

$$ \underset{N \times D}{x} \times \underset{D \times H}{W} = \underset{N \times H}{y} $$

形状のルール：掛ける行列の「内側」の次元 ($D$) が一致し、結果は「外側」の次元 ($N$ と $H$) になる。$N$ はデータ数 (バッチサイズ)、$D$ は入力の特徴数、$H$ は出力の特徴数に対応する。  

## 41.2 行列の積の逆伝播 ― 形状から導く

行列の積の逆伝播は「形状が合うように組み立てる」と理解しやすい。  

求めたいのは、$x$ と $W$ それぞれに関する微分 $\dfrac{\partial L}{\partial x}$, $\dfrac{\partial L}{\partial W}$ である ($L$ は最終的な損失というスカラ)。上流から来る微分 $gy = \dfrac{\partial L}{\partial y}$ は $y$ と同じ形状 $N \times H$ になる。  

$x$ の微分：$\dfrac{\partial L}{\partial x}$ は $x$ と同じ形状 $N \times D$ になる。手元にあるのは $gy$ ($N \times H$) と $W$ ($D \times H$)。これらを掛けて $N \times D$ を作るには：

$$ \frac{\partial L}{\partial x} = \underset{N \times H}{gy} \times \underset{H \times D}{W^\top} = \underset{N \times D}{\phantom{x}} \qquad(W^\top \text{は } W \text{の転置})$$

$W$ の微分：$\dfrac{\partial L}{\partial W}$ は $W$ と同じ形状 $D \times H$ になる。手元の $x$（$N \times D$）と $gy$（$N \times H$）から $D \times H$ を作るには：

$$ \frac{\partial L}{\partial W} = \underset{D \times N}{x^\top} \times \underset{N \times H}{gy} = \underset{D \times H}{\phantom{W}} $$

> まとめ：「微分は元の変数と同じ形状になる」というルールを使い、手元のベクトルや行列 ($gy$, $x$, $W$) を、形が合うように掛け合わせる。すると転置 $W^\top$, $x^\top$ が自然に現れる。  

In [ ]:
class MatMul(Function):
    def forward(self, x, W):
        y = x.dot(W)        # 行列の積 (NumPy の dot)
        return y
    def backward(self, gy):
        x, W = self.inputs
        # 形状が合うように組み立てる (前述の導出のとおり)
        gx = matmul(gy, W.T)    # gy(N,H) × W^T(H,D) = (N,D)  ← x と同じ形状
        gW = matmul(x.T, gy)    # x^T(D,N) × gy(N,H) = (D,H)  ← W と同じ形状
        return gx, gW

def matmul(x, W):
    return MatMul()(x, W)

# 動作確認：x(2x3) × W(3x4) = y(2x4)
x = Variable(np.random.randn(2, 3))
W = Variable(np.random.randn(3, 4))
y = matmul(x, W)
print("x：", x.shape, " W：", W.shape, " → y：", y.shape)

y.backward()
print("x.grad の形状：", x.grad.shape, " ← x と同じ (2,3)")
print("W.grad の形状：", W.grad.shape, " ← W と同じ (3,4)")


形状が正しく保たれている。逆伝播が「形の合う掛け算」で実現できていることになる。  

## 41.3 アフィン変換 (linear) ― 全結合層

ニューラルネットワークの全結合層は、行列の積にバイアス $b$ を足したも。  

$$ y = xW + b $$

これをアフィン変換とも呼ぶ。$b$ を足すことで、より柔軟な変換ができる。  

バイアスの逆伝播：$b$ は各データに足される (ブロードキャスト)。だから $b$ の微分は、ステップ 40 で学んだとおり `sum_to` で $b$ の形状に集約する。  

`linear` 関数として実装する (バイアス `b` は省略可能に)。  


In [ ]:
class Linear(Function):
    def forward(self, x, W, b):
        y = x.dot(W)
        if b is not None:
            y += b           # バイアスを足す (ブロードキャスト)
        return y
    def backward(self, gy):
        x, W, b = self.inputs
        # b の微分：gy を b の形状に集約 (ブロードキャストの逆)
        gb = None if b.data is None else sum_to(gy, b.shape)
        gx = matmul(gy, W.T)     # MatMul と同じ
        gW = matmul(x.T, gy)
        return gx, gW, gb

def linear(x, W, b=None):
    return Linear()(x, W, b)

# 動作確認: y = xW + b
x = Variable(np.random.randn(2, 3))    # データ 2 個、特徴3次元
W = Variable(np.random.randn(3, 4))    # 3 次元 → 4 次元へ変換
b = Variable(np.random.randn(4))       # バイアス (出力 4 次元ぶん)
y = linear(x, W, b)
print("y = xW + b の形状：", y.shape, " ← (2, 4)")

y.backward()
print("x.grad：", x.grad.shape, " W.grad：", W.grad.shape, " b.grad：", b.grad.shape)


$y = xW + b$ という全結合層の計算と、その逆伝播 (各パラメータの勾配) が正しく求まった。これはニューラルネットワークの最も基本的な部品にあたる。DeZero でニューラルネットワークが作れる準備が整った。  

> ステップ 41 のまとめ
>
> - 行列の積 `matmul` を実装。逆伝播は「形が合うように組み立てる」→ $gy W^\top$, $x^\top gy$。  
>
> - アフィン変換 `linear`（$y=xW+b$）＝ 全結合層を実装。
>
> - バイアスの逆伝播は `sum_to` で形状を集約。
>
> - これでニューラルネットワークの基本部品が揃った。


### DeZero で線形回帰をやってみる  

作った `matmul`/`linear` (や四則演算) を使い、線形回帰を DeZero で実際に学習させる。  

問題設定：ノイズを含むデータ $(x, y)$ に、直線 $y = Wx + b$ をあてはめる。「予測と正解の差の 2 乗の平均 (平均二乗誤差, MSE)」を損失とし、勾配降下法で $W, b$ を更新する。  

$$ \text{loss} = \frac{1}{N}\sum_{i=1}^{N}(y_i^{\text{pred}} - y_i)^2 $$

小さなデータ・少ないループで、3 時間の輪読会でもすぐ終わります。


In [ ]:
# mean (平均) を用意 (sum を要素数で割るだけ)
def mean_squared_error(x0, x1):
    diff = x0 - x1
    return sum(diff ** 2) / len(diff)

# --- データ生成：y = 2x + 5 + ノイズ ---
np.random.seed(0)
N = 100
x_data = np.random.rand(N, 1)                       # 0〜1 の入力
y_data = 2 * x_data + 5 + 0.1 * np.random.randn(N, 1)  # 正解は傾き 2・切片 5

x = Variable(x_data)
y = Variable(y_data)

# --- パラメータ初期化 ---
W = Variable(np.zeros((1, 1)))   # 傾き (最初は 0)
b = Variable(np.zeros(1))        # 切片 (最初は 0)

def predict(x):
    return matmul(x, W) + b      # y = xW + b

# --- 学習ループ ---
lr = 0.1
iters = 100
loss_history = []

for i in range(iters):
    y_pred = predict(x)
    loss = mean_squared_error(y, y_pred)

    W.cleargrad()
    b.cleargrad()
    loss.backward()

    # 勾配降下法で更新
    W.data -= lr * W.grad.data
    b.data -= lr * b.grad.data
    loss_history.append(float(loss.data))

print(f"学習後：W = {W.data[0,0]:.3f} (正解 2.0),  b = {b.data[0]:.3f} (正解 5.0)")
print(f"最終 loss = {loss_history[-1]:.4f}")


傾き $W \approx 2$、切片 $b \approx 5$ と、正解に近い値が学習できました。今回作ったテンソル関数だけで、機械学習の基本である線形回帰が動いたのです。結果を可視化します。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 左：データと学習した直線
axes[0].scatter(x_data, y_data, alpha=0.5, s=20, label='data')
xs = np.array([[0.0], [1.0]])
ys = xs * W.data[0, 0] + b.data[0]
axes[0].plot(xs, ys, 'r-', lw=2, label=f'fitted: y={W.data[0,0]:.2f}x+{b.data[0]:.2f}')
axes[0].set_xlabel('x'); axes[0].set_ylabel('y')
axes[0].set_title('Linear regression with DeZero')
axes[0].legend(); axes[0].grid(alpha=0.3)

# 右: 損失の推移
axes[1].plot(loss_history)
axes[1].set_xlabel('iteration'); axes[1].set_ylabel('loss (MSE)')
axes[1].set_title('Loss curve')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("左：赤い直線がデータにフィット。右：損失が下がって収束している")


左図で赤い直線がデータにきれいにフィットし、右図で損失が下がって収束している。DeZero のテンソル関数が、実際の機械学習タスクで機能することを確かめた。  


---
# PyTorch での書き方

今回実装したテンソル関数はもちろん PyTorch にも同じ機能がある。DeZero で自作したものが PyTorch のどの関数に当たるかを見る。  

> この環境に PyTorch が無い場合、下のセルは自動でスキップして説明だけ表示する。  


In [ ]:
import torch

# reshape / transpose
x = torch.tensor([[1., 2., 3.], [4., 5., 6.]], requires_grad=True)
print("[reshape] x.reshape(6):", x.reshape(6).shape)
print("[transpose] x.T:", x.T.shape)

# sum
print("[sum] x.sum():", x.sum().item())
print("[sum axis] x.sum(dim=0):", x.sum(dim=0).tolist())

# matmul / linear
a = torch.randn(2, 3, requires_grad=True)
W = torch.randn(3, 4, requires_grad=True)
b = torch.randn(4, requires_grad=True)
y = a @ W + b                    # @ は行列積。DeZero の linear と同じ
print("[linear] (a @ W + b).shape:", tuple(y.shape))

# 逆伝播も同じ
y.sum().backward()
print("[backward] a.grad.shape:", tuple(a.grad.shape), " W.grad.shape:", tuple(W.grad.shape))


`torch.nn.functional.linear(x, W, b)` という専用関数もある (DeZero の `linear` と同じ)。

### DeZero と PyTorch の対応表 (第 4 ステージ前半)

| 機能 | DeZero | PyTorch |
|---|---|---|
| 形状変更 | `x.reshape(...)` | `x.reshape(...)` / `x.view(...)` |
| 転置 | `x.T` | `x.T` / `x.t()` |
| 合計 | `x.sum(axis=...)` | `x.sum(dim=...)` |
| ブロードキャスト | 四則演算で自動 + `sum_to` | 自動 (内部で同様の処理) |
| 行列積 | `matmul(x, W)` | `x @ W` / `torch.matmul` |
| 全結合層 | `linear(x, W, b)` | `F.linear(x, W, b)` / `nn.Linear` |

DeZero を自作すると、PyTorch の `x @ W` や `nn.Linear` の裏で何が起きているか (特に逆伝播で転置が現れる理由やブロードキャストの微分が `sum` になる理由) の理解の助けになる。  


---
# まとめ ― 第 4 ステージ前半 (テンソルへの対応)

ステップ 37〜41 を通して、DeZero はテンソル（多次元配列）を扱えるフレームワークになった。ニューラルネットワークを作る準備が完了した。  

1. ステップ 37 (テンソル)：これまでのスカラ用実装が要素ごとの計算ならテンソルでもそのまま動くことを確認した。微分は入力と同じ形状。
2. ステップ 38 (形状変更)：`reshape` (逆伝播は元の形に戻す)、`transpose` (逆伝播はもう一度転置) を実装。
3. ステップ 39 (和)：`sum` を実装。逆伝播は「全要素にばらまく」＝ `broadcast_to`。
4. ステップ 40 (ブロードキャスト)：`broadcast_to` と `sum_to` が逆の関係。四則演算をブロードキャスト対応になった (`sum_to` で形を戻す)。
5. ステップ 41 (行列積・アフィン変換)：`matmul` と `linear` ($y=xW+b$) を実装。逆伝播は「形が合うように組み立てる」。

### 各操作の順伝播と逆伝播
| 順伝播 | 逆伝播 |
|---|---|
| `reshape` | `reshape` (元の形へ) |
| `transpose` | `transpose` |
| `sum` (和) | `broadcast_to` (ばらまき) |
| `broadcast_to` (ばらまき) | `sum_to` (和) |
| `matmul` ($xW$) | `matmul` ($gyW^\top$, $x^\top gy$) |

「和」と「ばらまき」の対応関係が今回の要点。  

### 次のステップの内容 (ステップ 42 以降)
次回は、今回作った部品を組み合わせて本格的なニューラルネットワークを作っていく。具体的には、非線形な活性化関数 (シグモイドなど)、複数の層を重ねる仕組み (`Layer`/`Model` クラス)、パラメータをまとめて更新する最適化手法 (`Optimizer`) などを追加し、最終的には手書き数字認識 (MNIST) のような実際のタスクを解けるようにしていく。  